In [ ]:
import sympy as sp
from sympy import symbols, Matrix, zeros, I, Eq
from IPython.display import display, Math

# Coordinates and frequency
x, y, z = symbols('x y z', real=True)
omega = sp.Symbol(r'\omega', real=True)

# Derivative operators: use LaTeX-style names
dx = sp.Symbol(r'\partial_x', commutative=False)
dy = sp.Symbol(r'\partial_y', commutative=False)
dz = sp.Symbol(r'\partial_z', commutative=False)

# Anisotropic permittivity tensor elements
eps_labels = {
    ('x','x'): r'\varepsilon_{xx}', ('x','y'): r'\varepsilon_{xy}', ('x','z'): r'\varepsilon_{xz}',
    ('y','x'): r'\varepsilon_{yx}', ('y','y'): r'\varepsilon_{yy}', ('y','z'): r'\varepsilon_{yz}',
    ('z','x'): r'\varepsilon_{zx}', ('z','y'): r'\varepsilon_{zy}', ('z','z'): r'\varepsilon_{zz}',
}
eps = Matrix(3, 3, lambda i, j: sp.Symbol(eps_labels[('xyz'[i], 'xyz'[j])], commutative=False))

# Curl operator matrix K such that K * F = curl(F)
K = Matrix([
    [   0, -dz,  dy],
    [  dz,   0, -dx],
    [ -dy,  dx,   0],
])

# Zero and identity 3x3 blocks
O = zeros(3, 3)
Id = sp.eye(3)

# Build A and B for the generalized eigenvalue problem:
#   A * psi = i*omega * B * psi
A = Matrix.vstack(
    Matrix.hstack(O, -K),
    Matrix.hstack(K,  O),
)

B = Matrix.vstack(
    Matrix.hstack(eps, O),
    Matrix.hstack(O,   Id),
)

# Field vector components
E_x, E_y, E_z = symbols(r'E_x E_y E_z')
H_x, H_y, H_z = symbols(r'H_x H_y H_z')
psi = Matrix([E_x, E_y, E_z, H_x, H_y, H_z])

lhs = A * psi
rhs = I * omega * (B * psi)

# Uniform rendering
display(Math(r"A = " + sp.latex(A)))
display(Math(r"\varepsilon = " + sp.latex(eps)))
display(Math(r"B = " + sp.latex(B)))
display(Math(r"\psi = " + sp.latex(psi)))
display(Math(sp.latex((lhs - rhs))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [2]:
# Set partial_y = 0 (no y-dependence)
A0 = A.subs(dy, 0)
B0 = B.copy()
psi0 = psi.copy()

lhs0 = A0 * psi0
rhs0 = I * omega * (B0 * psi0)

display(Math(r"\text{With } \partial_y = 0:"))
display(Math(r"\mathbf{A} = " + sp.latex(A0)))
display(Math(r"\mathbf{B} = " + sp.latex(B0)))
display(Math(r"\boldsymbol{\psi} = " + sp.latex(psi0)))
display(Math(sp.latex(lhs0 - rhs0)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [3]:
# Permutation to basis [Ex, Ey, Hx, -Hy, Ez, Hz]
P = sp.Matrix([
    [1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0,-1, 0],
    [0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 1],
])

Ap = P * A0 * P.inv()
Bp = P * B0 * P.inv()
psip = P * psi0

lhsp = Ap * psip
rhsp = I * omega * (Bp * psip)

display(Math(r"\text{Permuted basis: } [E_x,\, E_y,\, H_x,\, -H_y,\, E_z,\, H_z]"))
display(Math(r"\mathbf{A}' = " + sp.latex(Ap)))
display(Math(r"\mathbf{B}' = " + sp.latex(Bp)))
display(Math(r"\boldsymbol{\psi}' = " + sp.latex(psip)))
display(Math(sp.latex(Eq(lhsp, rhsp))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [4]:
# Form M = A' - i*omega*B' so that M * psi' = 0
M = Ap - I * omega * Bp
M = sp.simplify(M)

display(Math(r"\mathbf{M} = \mathbf{A}' - i\omega\,\mathbf{B}' = " + sp.latex(M)))

# Block decomposition: top-left 4x4, top-right 4x2, bottom-left 2x4, bottom-right 2x2
M11 = M[:4, :4]
M12 = M[:4, 4:]
M21 = M[4:, :4]
M22 = M[4:, 4:]

display(Math(r"\mathbf{M}_{11} = " + sp.latex(M11)))
display(Math(r"\mathbf{M}_{12} = " + sp.latex(M12)))
display(Math(r"\mathbf{M}_{21} = " + sp.latex(M21)))
display(Math(r"\mathbf{M}_{22} = " + sp.latex(M22)))

# Schur complement: S = M11 - M12 * M22^{-1} * M21
M22_inv = M22.inv(method="LU")
S = M11 - M12 * M22_inv * M21
S = sp.simplify(S)

display(Math(r"\text{Schur complement (tracing out } E_z, H_z \text{):}"))
display(Math(r"\mathbf{S} = \mathbf{M}_{11} - \mathbf{M}_{12}\,\mathbf{M}_{22}^{-1}\,\mathbf{M}_{21} = " + sp.latex(S)))

# Reduced equation for transverse components
psi_t = psip[:4, :]
display(Math(r"\mathbf{S}\,\boldsymbol{\psi} = 0, \quad \boldsymbol{\psi} = " + sp.latex(psi_t)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
# Add explicit dependence: eps(x), fields(x,z)
eps_funcs = {}
for r in 'xyz':
    for c in 'xyz':
        sym = sp.Symbol(eps_labels[(r, c)], commutative=False)
        eps_funcs[sym] = sp.Function(eps_labels[(r, c)], commutative=False)(x)

field_funcs = {
    E_x: sp.Function('E_x')(x, z),
    E_y: sp.Function('E_y')(x, z),
    E_z: sp.Function('E_z')(x, z),
    H_x: sp.Function('H_x')(x, z),
    H_y: sp.Function('H_y')(x, z),
    H_z: sp.Function('H_z')(x, z),
}

# Substitute into S and psi_t
S_explicit = S.subs(eps_funcs)
psi_t_explicit = psi_t.subs(field_funcs)

display(Math(r"\text{With explicit dependence: } \varepsilon_{ij}(x),\; \mathbf{F}(x,z)"))
display(Math(r"\mathbf{S}(x) = " + sp.latex(S_explicit)))
display(Math(r"\boldsymbol{\psi}(x,z) = " + sp.latex(psi_t_explicit)))

eq_explicit = S_explicit * psi_t_explicit
eq_explicit = sp.simplify(eq_explicit)
display(Math(r"\mathbf{S(x)}\,\boldsymbol{\psi} = 0:"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
EzHz_matrix = sp.simplify(-M22_inv * M21)

EzHz_matrix_explicit = EzHz_matrix.subs(eps_funcs) if 'eps_funcs' in globals() else EzHz_matrix
label = r"-\mathbf{M}_{22}^{-1}\,\mathbf{M}_{21}(x) = " if 'eps_funcs' in globals() else r"-\mathbf{M}_{22}^{-1}\,\mathbf{M}_{21} = "
display(Math(label + sp.latex(EzHz_matrix_explicit)))


<IPython.core.display.Math object>

In [7]:
# Exchange matrix X (4x4 anti-diagonal identity)
X = sp.Matrix([
    [0, 0, 0, 1],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [1, 0, 0, 0],
])

# Verify X @ X = I
assert X * X == sp.eye(4), "X is not an involution!"

# S(x) @ X @ X @ psi_t = 0  =>  S_tilde @ psi_tilde = 0
# where S_tilde = S @ X,  psi_tilde = X @ psi_t
S_tilde = sp.simplify(S_explicit * X)
psi_tilde = X * psi_t_explicit

display(Math(r"\mathbf{X} = " + sp.latex(X) + r", \quad \mathbf{X}^2 = \mathbf{I}"))
display(Math(r"\mathbf{S}(x)\,\boldsymbol{\psi} = \mathbf{S}(x)\,\mathbf{X}\,\mathbf{X}\,\boldsymbol{\psi} = \widetilde{\mathbf{S}}(x)\,\widetilde{\boldsymbol{\psi}} = 0"))
display(Math(r"\widetilde{\mathbf{S}}(x) = \mathbf{S}(x)\,\mathbf{X} = " + sp.latex(S_tilde)))
display(Math(r"\widetilde{\boldsymbol{\psi}} = \mathbf{X}\,\boldsymbol{\psi} = " + sp.latex(psi_tilde)))

eq_tilde = sp.simplify(S_tilde * psi_tilde)
display(Math(r"\widetilde{\mathbf{S}}\,\widetilde{\boldsymbol{\psi}} = 0:"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:

EzHz_matrix_explicit = EzHz_matrix.subs(eps_funcs) if 'eps_funcs' in globals() else EzHz_matrix
label = r"-\mathbf{M}_{22}^{-1}\,\mathbf{M}_{21}(x) X = " if 'eps_funcs' in globals() else r"-\mathbf{M}_{22}^{-1}\,\mathbf{M}_{21} = "
EzHz_matrix_explicit_same_basis = EzHz_matrix_explicit * X
display(Math(label + sp.latex(EzHz_matrix_explicit_same_basis)))


<IPython.core.display.Math object>

In [9]:
S_rest = sp.simplify(S_tilde + dz * sp.eye(4))

display(Math(r"\mathbf{S}_{\mathrm{rest}} = " + sp.latex(S_rest)))


Q = sp.simplify(S_rest)

display(Math(r"\partial_z \widetilde{\boldsymbol{\psi}} = \mathbf{Q}(x,\partial_x,\omega)\,\widetilde{\boldsymbol{\psi}}"))
display(Math(r"\mathbf{Q} = " + sp.latex(Q)))

# Final equation with explicit dependence
Q_explicit = Q.subs(eps_funcs)
display(Math(r"\partial_z \widetilde{\boldsymbol{\psi}}(x,z) = \mathbf{Q}(x)\,\widetilde{\boldsymbol{\psi}}(x,z):"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
# Build Ez, Hz from the reduced field basis phi_tilde = [-Hy, Hx, Ey, Ex]^T,
# then construct D and the change-of-basis matrix P sending
# [-Hy, Hx, Ey, Ex]^T -> [-Hy, Hx, Ey, Dx]^T.

phi_tilde = psi_tilde
EzHz_from_phi_tilde = sp.simplify(EzHz_matrix_explicit_same_basis * phi_tilde)
Ez_from_phi_tilde = sp.simplify(EzHz_from_phi_tilde[0, 0])
Hz_from_phi_tilde = sp.simplify(EzHz_from_phi_tilde[1, 0])

display(Math(r"\widetilde{\boldsymbol{\phi}}(x,z) = " + sp.latex(phi_tilde)))
display(Math(r"\begin{bmatrix}E_z\\H_z\end{bmatrix} = " + sp.latex(EzHz_from_phi_tilde)))

def eps_func_entry(r, c):
    return eps_funcs[sp.Symbol(eps_labels[(r, c)], commutative=False)]

eps_xx_x = eps_func_entry('x', 'x')
eps_xy_x = eps_func_entry('x', 'y')
eps_xz_x = eps_func_entry('x', 'z')
eps_yx_x = eps_func_entry('y', 'x')
eps_yy_x = eps_func_entry('y', 'y')
eps_yz_x = eps_func_entry('y', 'z')
eps_zx_x = eps_func_entry('z', 'x')
eps_zy_x = eps_func_entry('z', 'y')
eps_zz_x = eps_func_entry('z', 'z')

minus_Hy_tilde = phi_tilde[0, 0]
Hx_tilde = phi_tilde[1, 0]
Ey_tilde = phi_tilde[2, 0]
Ex_tilde = phi_tilde[3, 0]

Dx_from_phi_tilde = sp.simplify(eps_xx_x * Ex_tilde + eps_xy_x * Ey_tilde + eps_xz_x * Ez_from_phi_tilde)
Dy_from_phi_tilde = sp.simplify(eps_yx_x * Ex_tilde + eps_yy_x * Ey_tilde + eps_yz_x * Ez_from_phi_tilde)
Dz_from_phi_tilde = sp.simplify(eps_zx_x * Ex_tilde + eps_zy_x * Ey_tilde + eps_zz_x * Ez_from_phi_tilde)

display(Math(r"D_x = " + sp.latex(Dx_from_phi_tilde)))
display(Math(r"D_y = " + sp.latex(Dy_from_phi_tilde)))
display(Math(r"D_z = " + sp.latex(Dz_from_phi_tilde)))

Dx_row = sp.simplify(sp.Matrix([[0, 0, eps_xy_x, eps_xx_x]]) + eps_xz_x * EzHz_matrix_explicit_same_basis[:1, :])

P_D = sp.Matrix([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [Dx_row[0, 0], Dx_row[0, 1], Dx_row[0, 2], Dx_row[0, 3]],
])

display(Math(r"\mathbf{P}(x) = " + sp.latex(P_D)))

Phi_tilde = sp.simplify(P_D * phi_tilde)
display(Math(r"\widetilde{\boldsymbol{\Phi}}(x,z) = \mathbf{P}(x)\,\widetilde{\boldsymbol{\phi}}(x,z) = " + sp.latex(Phi_tilde)))

d_entry = sp.simplify(Dx_row[0, 3])
d_inv = sp.Pow(d_entry, -1, evaluate=True)
P_D_inv = sp.Matrix([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [-d_inv * Dx_row[0, 0], -d_inv * Dx_row[0, 1], -d_inv * Dx_row[0, 2], d_inv],
])

Q_D = sp.simplify(P_D * Q_explicit * P_D_inv)

display(Math(r"\partial_z \widetilde{\boldsymbol{\phi}}(x,z) = \mathbf{Q}(x)\,\widetilde{\boldsymbol{\phi}}(x,z)"))
display(Math(r"\partial_z \widetilde{\boldsymbol{\Phi}}(x,z) = \mathbf{P}(x)\,\mathbf{Q}(x)\,\mathbf{P}^{-1}(x)\,\widetilde{\boldsymbol{\Phi}}(x,z)"))
display(Math(r"\mathbf{Q}_D(x) = \mathbf{P}(x)\,\mathbf{Q}(x)\,\mathbf{P}^{-1}(x) = " + sp.latex(sp.simplify(Q_D))))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [13]:
sp.latex(sp.simplify(Q_D))

'\\left[\\begin{matrix}- (\\left(- \\varepsilon_{xx}{\\left(x \\right)} + \\varepsilon_{xz}{\\left(x \\right)} \\varepsilon_{zz}^{-1}{\\left(x \\right)} \\varepsilon_{zx}{\\left(x \\right)}\\right) \\left(\\varepsilon_{xx}{\\left(x \\right)} - \\varepsilon_{xz}{\\left(x \\right)} \\varepsilon_{zz}^{-1}{\\left(x \\right)} \\varepsilon_{zx}{\\left(x \\right)}\\right)^{-1} \\varepsilon_{xz}{\\left(x \\right)} \\varepsilon_{zz}^{-1}{\\left(x \\right)} \\partial_{x} + \\varepsilon_{xz}{\\left(x \\right)} \\varepsilon_{zz}^{-1}{\\left(x \\right)} \\partial_{x}) & 0 & i \\omega \\left(- \\left(- \\varepsilon_{xx}{\\left(x \\right)} + \\varepsilon_{xz}{\\left(x \\right)} \\varepsilon_{zz}^{-1}{\\left(x \\right)} \\varepsilon_{zx}{\\left(x \\right)}\\right) \\left(\\varepsilon_{xx}{\\left(x \\right)} - \\varepsilon_{xz}{\\left(x \\right)} \\varepsilon_{zz}^{-1}{\\left(x \\right)} \\varepsilon_{zx}{\\left(x \\right)}\\right)^{-1} \\left(\\varepsilon_{xy}{\\left(x \\right)} - \\varepsilon_{xz}{\\